# ✅ Checking whether an answer is actually right

*"What is the price of Romaine lettuce?"* and *"When does the Stockton SQF certificate
expire?"* are **document** questions. The answer is written in prose inside a chunk — it is
not a column you can `SELECT`. So you cannot verify it with a query like
`SELECT price FROM ...`.

What you can do — and what this notebook is — is use SQL to **go and read the source text
yourself**, then compare.

```
   the system's answer   ─┐
                          ├─►  do they match?
   the raw passage        ─┘
```

Three tools, then one command that does the whole comparison for you.

> Run the `sql()` helper from `Explore_Graph_SQL.ipynb` first (or the cell below, which
> defines it again). Everything here is read-only.

## 0 — Helpers

In [ ]:
def sql(query, params=None):
    with db(dict_rows=True) as cur:
        cur.execute(query, params)
        rows = cur.fetchall()
    return pd.DataFrame([dict(r) for r in rows]) if rows else pd.DataFrame()


def grep_docs(term, limit=20):
    """Every passage containing `term`, with the match highlighted between « ».

    ts_headline is PostgreSQL's own snippet-and-highlight function — the same machinery
    behind full-text search. You see the words around the hit, not the whole chunk.
    """
    return sql("""
        SELECT c.doc_id, d.title, c.chunk_id, c.chunk_index,
               ts_headline('english', c.content,
                           plainto_tsquery('english', %(q)s),
                           'MaxFragments=3, MinWords=6, MaxWords=28, '
                           'StartSel=«, StopSel=», FragmentDelimiter= … ') AS match
        FROM rag_chunks c
        JOIN rag_documents d ON d.doc_id = c.doc_id
        WHERE c.content ILIKE %(like)s
        ORDER BY d.title, c.chunk_index
        LIMIT %(lim)s
    """, {"q": term, "like": f"%{term}%", "lim": limit})


def sentences_with(term, limit=40):
    """Just the sentences that mention the term. Usually all you need to read."""
    return sql("""
        SELECT d.title, btrim(s.sentence) AS sentence
        FROM rag_chunks c
        JOIN rag_documents d ON d.doc_id = c.doc_id
        CROSS JOIN LATERAL regexp_split_to_table(c.content, E'\\.\\s+|\\n') AS s(sentence)
        WHERE s.sentence ILIKE %(like)s
        ORDER BY d.title
        LIMIT %(lim)s
    """, {"like": f"%{term}%", "lim": limit})


print("Ready: sql() · grep_docs() · sentences_with()")

## 1 — "What is the price of Romaine lettuce?"

Read the source. Whatever `«…»` shows here **is** the truth; if the system said something
else, the system is wrong.

In [ ]:
grep_docs("Romaine")

### 1b — Just the sentences

In [ ]:
sentences_with("Romaine")

### 1c — Pull the numbers out

Every price-shaped string in a passage that mentions the term. If the system quoted a figure
that is not in this list, it invented it — or borrowed it from a neighbouring product.

In [ ]:
sql("""
SELECT DISTINCT d.title, m[1] AS price_found
FROM rag_chunks c
JOIN rag_documents d ON d.doc_id = c.doc_id
CROSS JOIN LATERAL regexp_matches(
        c.content,
        '(\\$\\s?[0-9][0-9,]*\\.?[0-9]*(?:\\s*(?:per|/)\\s*[a-zA-Z]+)?)', 'g') AS m
WHERE c.content ILIKE %(like)s
ORDER BY d.title, price_found
""", {"like": "%Romaine%"})

## 2 — "When does the Stockton SQF certificate expire?"

Same method, different shape of fact. Search for the *facility*, not the certificate id —
the register is likely to write "Stockton" in prose and the id only in a column.

In [ ]:
grep_docs("Stockton")

### 2b — Every date near a mention of Stockton

Handles `2025-06-30`, `30 June 2025` and `June 30, 2025`.

In [ ]:
sql("""
SELECT DISTINCT d.title, m[1] AS date_found
FROM rag_chunks c
JOIN rag_documents d ON d.doc_id = c.doc_id
CROSS JOIN LATERAL regexp_matches(
        c.content,
        '([0-9]{4}-[0-9]{2}-[0-9]{2}|[0-9]{1,2}\\s+[A-Z][a-z]+\\s+[0-9]{4}|[A-Z][a-z]+\\s+[0-9]{1,2},\\s*[0-9]{4})',
        'g') AS m
WHERE c.content ILIKE %(like)s
ORDER BY d.title, date_found
""", {"like": "%Stockton%"})

### 2c — Narrow it to the certification register

If several documents mention Stockton, the register is the authority on expiry dates.

In [ ]:
sentences_with("expir")

## 3 — `audit()` — do it all in one call

Asks the question, then goes and reads the source, then tells you whether the system cited
the documents that actually contain the term.

**The check that matters is the last line.** If a document contains your term but was never
cited, the answer was written without seeing it — which is exactly how a confident, incomplete
answer happens.

In [ ]:
def audit(question, term, mode="rag"):
    """Ask, then verify against the raw text."""
    print("=" * 78)
    print("1 · THE SYSTEM'S ANSWER")
    print("=" * 78)
    res = ask(question, mode=mode)

    print("\n" + "=" * 78)
    print(f"2 · THE SOURCE — every passage containing {term!r}")
    print("=" * 78)
    truth = grep_docs(term)
    display(truth)

    print("=" * 78)
    print("3 · DID IT READ THE RIGHT DOCUMENTS?")
    print("=" * 78)
    cited = {s["doc_id"] for s in (res.get("sources") or [])} if isinstance(res, dict) else set()
    real  = set(truth["doc_id"]) if len(truth) else set()
    print("  cited            :", ", ".join(sorted(cited)) or "(none)")
    print("  actually mentions:", ", ".join(sorted(real)) or "(none)")

    missed = real - cited
    extra  = cited - real
    if not real:
        print("\n  ⚠️  NOTHING in the corpus contains that term.")
        print("     The correct answer is a refusal. If you got a figure, it was invented.")
    elif missed:
        print(f"\n  ⚠️  {len(missed)} document(s) mention it but were NOT cited:",
              ", ".join(sorted(missed)))
        print("     The answer may be incomplete. Try mode=\"graph\", or raise RERANK_TOP_N.")
    else:
        print("\n  ✅ every document containing the term was cited.")
    if extra:
        print("  (cited without containing the exact term — normal: retrieval is by meaning,",
              "\n   not by string. Worth a glance if the answer looks wrong.)")
    return {"answer": res, "truth": truth, "cited": cited, "mentions": real}


print("Ready: audit(question, term)")

## 3b — Try it

In [ ]:
audit("What is the price quoted for Romaine lettuce?", "Romaine")

In [ ]:
audit("When does the Stockton SQF certificate expire, and what happens if it lapses?",
      "Stockton", mode="graph")

## 4 — Verifying a *graph* answer

For a relationship claim ("Tidewater's contract requires the Stockton certificate"), the
ground truth is an **edge**, not a passage. This asks the graph directly.

If the system asserted a relationship that does not appear here, it was invented — the model
is supposed to cite these as `[graph]` and nothing else.

In [ ]:
sql("""
SELECT s.label AS subject, e.predicate AS verb, t.label AS object, e.source
FROM rag_edges e
JOIN rag_nodes s ON s.node_id = e.src_id
JOIN rag_nodes t ON t.node_id = e.dst_id
WHERE s.label ILIKE %(x)s OR t.label ILIKE %(x)s
   OR s.node_id ILIKE %(x)s OR t.node_id ILIKE %(x)s
ORDER BY e.predicate
""", {"x": "%Stockton%"})

## 5 — When the answer is "I don't have that"

Before trusting a refusal, check the corpus really is silent. If this returns rows, the
information *is* there and retrieval missed it — a very different bug from a correct refusal.

In [ ]:
term = "carbon footprint"          # ⬅️ change this
hits = grep_docs(term)
if len(hits):
    print(f"⚠️  {len(hits)} passage(s) DO mention {term!r} — the refusal was wrong.")
    display(hits)
else:
    print(f"✅ nothing in the corpus mentions {term!r} — refusing was correct.")

## 6 — The habit worth keeping

Three questions, in this order, whenever an answer looks off:

1. **Is it in the corpus at all?** → `grep_docs(term)`. Empty means a refusal was right.
2. **Did the system read the passage that contains it?** → `audit(question, term)`. A
   document that mentions the term but was never cited is the usual cause of a confident,
   incomplete answer.
3. **For relationship claims, does the edge exist?** → the query in Section 4. No edge means
   the model invented the connection.

Most "the AI is wrong" reports resolve at step 1 or 2, and neither is a model problem.